In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display


%load_ext autoreload
%autoreload 2

from vpop_calibration import *

## Benchmarking Runtimes for Mavoglurant surrogate model

## Loading Reference Data

In [ ]:
df = pd.read_csv("../Mavoglurant_Dataset.csv")


obs_df = (
    df.loc[df["EVID"] == 0]
      .rename(
          columns={
              "ID": "id",
              "TIME": "time",
              "DV": "value",
              "DOSE": "dose"

          }
      )
      [["id", "time", "value", "dose", "WT"]]
      .astype({"value": "float"})
)
obs_df["time"] = obs_df["time"].apply(lambda t: t * 60 * 60)
obs_df["value"]= obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "logC15"
display(df.head())
display(obs_df.head())

In [ ]:
train_df = obs_df.copy()
train_df["protocol_arm"] = train_df["dose"].apply(lambda x: f"protocole-{float(x):.1f}")
train_df = train_df.drop(columns=["dose"])
display(train_df.head())

## Reference model

In [ ]:
model = SimworkModelBinding(
    path_to_model="cm.json",
    path_to_solving_options="sv.json",
    inputs=["KbBR", "CLint","KbMU","KbAD","KbBO","KbRB","WT","dose"],
    outputs=["logC15"],
)
print(model.inputs)

struct_model = StructuralSimwork(model=model)

## Generate training data

In [ ]:
protocol_design = pd.DataFrame({
    "protocol_arm": ["protocole-25.0","protocole-37.5","protocole-50.0"],"dose": [25,37.5,50.0] 
})
struct_model = StructuralSimwork(
    model=model,
    protocol_design=protocol_design,
)

param_ranges = {

    "WT": {
        "low": 50,
        "high": 110,
        "log": False,
    },

    "CLint": {
        "low": 6,
        "high": 8,
        "log": True,
    },

    "KbBR": {
        "low": -3,
        "high": 2,
        "log": True,
    },

    "KbMU": {
        "low": -2,
        "high": 3,
        "log": True,
    },

    "KbBO": {
        "low": -1,
        "high": 3,
        "log": True,
    },

    "KbAD": {
        "low": 1,
        "high": 2.5,
        "log": True,
    },

    "KbRB": {
        "low": -2,
        "high": 1.5,
        "log": True,
    },
}

In [ ]:
time_span = (0, 48*60*60)
nb_steps = 100
time_steps = np.linspace(*time_span, nb_steps).tolist()
dataset = generate_training_data(
    struct_model=struct_model,
    ranges=param_ranges,
    log_nb_ind=5,
    time=time_steps,
)

## Train surrogate

In [ ]:
def train_model(input_df: pd.DataFrame) -> GP:
    learned_ode_params = list(param_ranges.keys())
    descriptors = learned_ode_params + ["time"]
    gp = GP(
        input_df,
        descriptors=descriptors,
        var_strat="IMV",  # either IMV (Independent Multitask Variational) or LMCV (Linear Model of Coregionalization Variational)
        kernel="RBF",  # Either RBF or SMK
        nb_inducing_points=300,
        mll="ELBO",  # default, otherwise PLL
        nb_training_iter=1000,
        training_proportion=0.7,
        learning_rate=0.05,
        lr_decay=0.99,
        jitter=1e-6,
        log_inputs=learned_ode_params,
        log_outputs=[],
        plot_frame_rate=50,
        min_delta= 0.001,
    )
    gp.train()

    return gp

In [ ]:
%%capture output
%timeit -n 1 -r 5 -v time_train gp = train_model(dataset)

In [ ]:
print(time_train.timings)

In [ ]:
gp = train_model(dataset)
gp.plot_obs_vs_predicted(data_set="training")
gp.plot_obs_vs_predicted(data_set="validation")

## Optimize the GP surrogate using SAEM

In [ ]:
structural_gp = StructuralGp(gp)

In [ ]:
# NLME model parameters
prior_MI = {
    "model_intrinsic": {
        "KbBR": {"prior": np.exp(0.5),"constraint": structural_gp.training_ranges["KbBR"]},
        "KbMU": {"prior": np.exp(0.3),"constraint": structural_gp.training_ranges["KbMU"]},
        "KbBO": {"prior": np.exp(0.03),"constraint": structural_gp.training_ranges["KbBO"]},
        "KbAD":  {"prior": np.exp(2), "constraint": structural_gp.training_ranges["KbAD"]},
        "KbRB": {"prior": np.exp(0.3),"constraint": structural_gp.training_ranges["KbRB"]},
    },
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 0.5, "constraint": structural_gp.training_ranges["CLint"]},
    },
    "pdk": {
        "WT",
    },
    "error_model": {
        "logC15": {"error_type": "additive", "sigma": 0.5},
    },
}
prior_pdu = {
    "pdu": {
        "CLint": {"prior": np.exp(7.6), "prior_omega": 0.5, "constraint": structural_gp.training_ranges["CLint"]},
        "KbBR": {"prior": np.exp(0.5),  "prior_omega": 0.25,"constraint": structural_gp.training_ranges["KbBR"]},
        "KbMU": {"prior": np.exp(0.3),  "prior_omega": 0.25,"constraint": structural_gp.training_ranges["KbMU"]},
        "KbBO": {"prior": np.exp(0.03),  "prior_omega": 0.25,"constraint": structural_gp.training_ranges["KbBO"]},
        "KbAD":  {"prior": np.exp(2),  "prior_omega": 0.25, "constraint": structural_gp.training_ranges["KbAD"]},
        "KbRB": {"prior": np.exp(0.3),  "prior_omega": 0.25,"constraint": structural_gp.training_ranges["KbRB"]},
    },
    "pdk": {
        "WT",
    },
    "error_model": {
        "logC15": {"error_type": "additive", "sigma": 0.5},
    },
}

In [ ]:
def fit_saem(prior)-> NlmeModel:
    config = Config(
        saem=SaemConfigDict(
            nb_phase1_iterations=100,
            nb_phase2_iterations=100,
            plot_frames=5,
            optim_max_fun=2,
            verbose=False,
            mcmc_first_burn_in=0,
        ),
        nlme=NlmeConfigDict(nb_chains=1),
    )
    nlme_surrogate = NlmeModel(df=train_df, prior_params=prior, structural_model=structural_gp, config=config)
    nlme_surrogate.optimizer.run()
    return nlme_surrogate

In [ ]:
%%capture output
%timeit -n 1 -r 5 -v time_saem_MI fit_saem(prior_MI)

In [ ]:
perf_df = pd.DataFrame({
    "time_train": time_train.timings,
    "time_saem": time_saem_MI.timings,
})

perf_df["total_time"] = (
    perf_df["time_train"] +
    perf_df["time_saem"]
)

perf_df.to_csv(
    "Mavoglurant_Simwork_surrogate_MI_runtime.csv",
    index=False
)

perf_df

In [ ]:
%%capture output
%timeit -n 1 -r 5 -v time_saem_PDU fit_saem(prior_pdu)

In [ ]:
perf_df = pd.DataFrame({
    "time_train": time_train.timings,
    "time_saem": time_saem_PDU.timings,
})

perf_df["total_time"] = (
    perf_df["time_train"] +
    perf_df["time_saem"]
)

perf_df.to_csv(
    "Mavoglurant_Simwork_surrogate_PDU_runtime.csv",
    index=False
)

perf_df